# Full-map Forward Model Pipeline · κ → noisy/masked (γ₁, γ₂)

Build one reproducible, full-resolution observation per FITS convergence map using flat-sky Kaiser–Squires shear, Gaussian shape noise, and survey masks. Uses NumPy/SciPy/Astropy only for preprocessing; no LensTools or healpy.

- Generated `{stem}_input.npy` arrays store float32 `(3, H, W)` = `[γ₁ noisy+masked, γ₂ noisy+masked, mask]`.
- Generated `{stem}_target.npy` arrays store float32 `(H, W)` = κ, optionally with the **full-map** mean removed.
- The generated arrays are members of `processed_maps.tar`; `manifest.csv`, `qa_forward_model.png`, and `config_used.yaml` are saved beside it in `OUTPUT_DIR`.
- Masks/noise are generated once per source map and shared by the full-map, fixed-tile, and per-epoch random-crop loader views.
- `manifest.csv` has one row per source map, preserving line-of-sight grouping for later splits. The original source TAR is never modified.

**Sections:** 1 Imports & YAML configuration · 2 Physics functions and tests · 3 Noise & full-map mask · 4 Full-map pipeline and visual QA · 5 Batch saving and verification · 6 Dataset loader, crop modes, and shear-aware augmentation · Self-checks · Notes

## 1. Imports & Configuration

The settings live in one YAML file. Set `FORWARD_MODEL_CONFIG` to a teammate-specific YAML path, or edit `CONFIG_PATH_OVERRIDE` below. `DATASET_ROOT` is the one path teammates usually need to customize; relative `INPUT_DIR` and `OUTPUT_DIR` are resolved from it. The portable starter template is `configs/forward_model/example.yaml`; see this workflow folder's README for setup instructions.

The default Colab configuration targets the 120 FITS maps in the current sample folder; update `INPUT_DIR` when moving to the full extracted dataset. The input TAR is not scanned or changed. Generated arrays are packed into `processed_maps.tar` under `OUTPUT_DIR`, next to the manifest CSV and QA image. `PATCH_SIZE` controls QA grid lines and loader crops only—maps are stored whole.

In [ ]:
import csv
import hashlib
import io
import os
import re
import tarfile
import tempfile
import warnings
import zlib
from functools import lru_cache
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import yaml
from astropy.io import fits
from scipy import fft as sfft

try:                                   # optional nicer progress bar
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

print(f"numpy {np.__version__}")

In [ ]:
# Override with FORWARD_MODEL_CONFIG=/path/to/team_config.yaml, or set a path below.
CONFIG_PATH_OVERRIDE = None

DEFAULT_CONFIG_YAML = """
# Change DATASET_ROOT once for each machine/account; relative paths use this root.
DATASET_ROOT: /content/drive/MyDrive/Astronova/Research/DataSet
INPUT_DIR: Subset
OUTPUT_DIR: generated_data/sample_1
PATCH_SIZE: 256
NOISE_LEVEL: 0.05
MASK_FRACTION: 0.15
SEED: 42
N_QA_FILES: 4
MAX_FILES: null
STAR_RADIUS_PX: [3, 13]
GAP_HEIGHT_PX: [6, 12]
TARGET_ZERO_MEAN: true
QA_FIGURE: qa_forward_model.png
"""

config_override = CONFIG_PATH_OVERRIDE or os.environ.get("FORWARD_MODEL_CONFIG")
if config_override:
    CONFIG_PATH = Path(os.path.expandvars(os.path.expanduser(str(config_override))))
elif Path("/content/drive/MyDrive/Astronova/Research/DataSet").is_dir():
    CONFIG_PATH = Path("/content/drive/MyDrive/Astronova/Research/DataSet/config_forward_model.yaml")
else:
    CONFIG_PATH = Path.cwd() / "configs" / "forward_model" / "config.yaml"

default_cfg = yaml.safe_load(DEFAULT_CONFIG_YAML)
if not Path("/content/drive/MyDrive/Astronova/Research/DataSet").is_dir():
    default_cfg.update({
        "DATASET_ROOT": str(Path.cwd() / "data"),
        "INPUT_DIR": "raw_subset",
        "OUTPUT_DIR": "generated_data/sample_1",
    })
if not CONFIG_PATH.exists():
    CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
    CONFIG_PATH.write_text(yaml.safe_dump(default_cfg, sort_keys=False), encoding="utf-8")
    print(f"Created config: {CONFIG_PATH.resolve()} (edit it before running the pipeline if paths differ)")

with CONFIG_PATH.open(encoding="utf-8") as fh:
    loaded_cfg = yaml.safe_load(fh) or {}
cfg = {**default_cfg, **loaded_cfg}

DATASET_ROOT = Path(os.path.expandvars(os.path.expanduser(str(cfg["DATASET_ROOT"]))))
if not DATASET_ROOT.is_absolute():
    DATASET_ROOT = (CONFIG_PATH.parent / DATASET_ROOT).resolve()
else:
    DATASET_ROOT = DATASET_ROOT.resolve()


def resolve_config_path(value: str) -> Path:
    """Resolve configured paths relative to DATASET_ROOT, expanding ~ and environment variables."""
    path = Path(os.path.expandvars(os.path.expanduser(str(value))))
    return path.resolve() if path.is_absolute() else (DATASET_ROOT / path).resolve()


INPUT_DIR = resolve_config_path(cfg["INPUT_DIR"])
OUTPUT_DIR = resolve_config_path(cfg["OUTPUT_DIR"])
PATCH_SIZE = int(cfg["PATCH_SIZE"])
NOISE_LEVEL = float(cfg["NOISE_LEVEL"])
MASK_FRACTION = float(cfg["MASK_FRACTION"])
SEED = int(cfg["SEED"])
N_QA_FILES = int(cfg["N_QA_FILES"])
MAX_FILES = int(cfg["MAX_FILES"]) if cfg.get("MAX_FILES") is not None else None
STAR_RADIUS_PX = tuple(int(v) for v in cfg["STAR_RADIUS_PX"])
GAP_HEIGHT_PX = tuple(int(v) for v in cfg["GAP_HEIGHT_PX"])
TARGET_ZERO_MEAN = bool(cfg["TARGET_ZERO_MEAN"])
QA_FIGURE = str(cfg["QA_FIGURE"])

if PATCH_SIZE <= 0:
    raise ValueError("PATCH_SIZE must be positive")
if NOISE_LEVEL < 0:
    raise ValueError("NOISE_LEVEL must be >= 0")
if not 0 <= MASK_FRACTION < 1:
    raise ValueError("MASK_FRACTION must be in [0, 1)")
if N_QA_FILES <= 0:
    raise ValueError("N_QA_FILES must be positive")
if MAX_FILES is not None and MAX_FILES <= 0:
    raise ValueError("MAX_FILES must be a positive integer or null")
for name, bounds, minimum in (("STAR_RADIUS_PX", STAR_RADIUS_PX, 1), ("GAP_HEIGHT_PX", GAP_HEIGHT_PX, 1)):
    if len(bounds) != 2 or bounds[0] < minimum or bounds[0] > bounds[1]:
        raise ValueError(f"{name} must be [lo, hi] with {minimum} <= lo <= hi")
if not INPUT_DIR.is_dir():
    raise FileNotFoundError(f"INPUT_DIR does not exist: {INPUT_DIR}. Edit {CONFIG_PATH}")

FITS_FILES = sorted(str(p) for p in INPUT_DIR.rglob("*")
                    if p.is_file() and p.name.lower().endswith((".fits", ".fit")))
if MAX_FILES is not None:
    FITS_FILES = FITS_FILES[:MAX_FILES]
if not FITS_FILES:
    raise FileNotFoundError(f"No .fits/.fit files found recursively under {INPUT_DIR}")

cfg_used = {**cfg, "DATASET_ROOT_RESOLVED": str(DATASET_ROOT),
            "INPUT_DIR_RESOLVED": str(INPUT_DIR), "OUTPUT_DIR_RESOLVED": str(OUTPUT_DIR),
            "FOUND_MAP_COUNT": len(FITS_FILES)}
print(f"Config file : {CONFIG_PATH.resolve()}")
print(f"Input       : {INPUT_DIR} ({len(FITS_FILES)} FITS maps)")
print(f"Output      : {OUTPUT_DIR}")
print(f"PATCH_SIZE={PATCH_SIZE} NOISE_LEVEL={NOISE_LEVEL} MASK_FRACTION={MASK_FRACTION} SEED={SEED}")
print(f"STAR_RADIUS_PX={STAR_RADIUS_PX} GAP_HEIGHT_PX={GAP_HEIGHT_PX} TARGET_ZERO_MEAN={TARGET_ZERO_MEAN}")

### I/O and map helpers

`load_kappa` reads the primary HDU (falling back to the first 2-D image HDU) and converts NaN/±inf to zero with a warning. Filename metadata is retained in the one-row-per-map manifest.

In [ ]:
def load_kappa(path: str) -> np.ndarray:
    """Load a 2-D convergence map as native-endian float32; NaN/inf -> 0."""
    with fits.open(path, memmap=False) as hdul:
        data = hdul[0].data
        if data is None or np.ndim(data) != 2:
            for hdu in hdul[1:]:
                if hdu.data is not None and np.ndim(hdu.data) == 2:
                    data = hdu.data
                    break
            else:
                raise ValueError("no 2-D image data in any HDU")
        arr = np.array(data, dtype=np.float32)
    n_bad = int((~np.isfinite(arr)).sum())
    if n_bad:
        warnings.warn(f"{os.path.basename(path)}: {n_bad} non-finite pixels set to 0")
    return np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)


_Z_RE = re.compile(r"_z(\d+(?:\.\d+)?)", re.IGNORECASE)
_R_RE = re.compile(r"_(\d+)r(?=\.|_|$)", re.IGNORECASE)


def parse_metadata(path: str) -> dict:
    name = os.path.basename(path)
    z, r = _Z_RE.search(name), _R_RE.search(name)
    return {"source_fits": name,
            "redshift": float(z.group(1)) if z else float("nan"),
            "realization": r.group(1) if r else ""}


def patch_slices(shape, patch_size):
    """Non-overlapping tiling. A 512x512 map with 256 patches -> 4 tiles."""
    for iy in range(shape[0] // patch_size):
        for ix in range(shape[1] // patch_size):
            yield iy, ix, (slice(iy * patch_size, (iy + 1) * patch_size),
                           slice(ix * patch_size, (ix + 1) * patch_size))


def center_slices(shape, patch_size):
    if min(shape) < patch_size:
        raise ValueError(f"map {shape} smaller than patch size {patch_size}")
    y0, x0 = (shape[0] - patch_size) // 2, (shape[1] - patch_size) // 2
    return slice(y0, y0 + patch_size), slice(x0, x0 + patch_size)


def stable_seed_key(path: str) -> int:
    """Order-independent integer from the filename, so masks/noise do not change if the file list changes."""
    return zlib.crc32(os.path.basename(path).encode())

## 2. The Physics Functions (Forward Model)

Flat-sky Kaiser–Squires relations in Fourier space (with $l^2=l_x^2+l_y^2$):

$$\tilde\gamma_1=\frac{l_x^2-l_y^2}{l^2}\,\tilde\kappa,\qquad \tilde\gamma_2=\frac{2\,l_x l_y}{l^2}\,\tilde\kappa .$$

**Implementation notes**
- The DC mode is set to zero: a constant κ (mass sheet) produces no shear, so its mean cannot be reconstructed from shear.
- Frequencies use cycles/pixel; the pixel scale cancels in the kernel ratios.
- Real FFTs preserve real outputs. Nyquist row/column modes remain zeroed because the odd γ₂ kernel cannot represent them while preserving Hermitian symmetry.
- For every retained non-DC mode, $d_1^2+d_2^2=1$, which defines the inverse and tests below.
- Shear is always computed on the full input map before any crops are taken. This avoids applying periodic FFT boundaries to individual 256×256 tiles.
- Keep the γ₂ axis/sign convention consistent across the project.

In [ ]:
@lru_cache(maxsize=8)
def _nyquist_keep(shape: tuple[int, int]) -> np.ndarray:
    """1 everywhere on the rfft2 grid except the Nyquist row/column (even sizes), which are 0.

    The gamma2 kernel is odd in frequency, so on the Nyquist row/column it cannot respect the
    Hermitian symmetry of a real field and irfft2 would silently drop part of the signal.
    Zeroing those modes (a negligible fraction of the power) is the standard fix.
    """
    ny, nx = shape
    keep = np.ones((ny, nx // 2 + 1))
    if ny % 2 == 0:
        keep[ny // 2, :] = 0.0
    if nx % 2 == 0:
        keep[:, -1] = 0.0
    keep.setflags(write=False)
    return keep


@lru_cache(maxsize=8)
def _ks_kernels(shape: tuple[int, int]) -> tuple[np.ndarray, np.ndarray]:
    """Kaiser-Squires kernels d1=(lx^2-ly^2)/l^2 and d2=2 lx ly/l^2 on the rfft2 grid.
    DC mode and Nyquist row/column are set to 0."""
    ny, nx = shape
    ly = sfft.fftfreq(ny)[:, None]       # axis 0 = y
    lx = sfft.rfftfreq(nx)[None, :]      # axis 1 = x (half-plane for rfft2)
    l2 = lx**2 + ly**2
    l2_safe = np.where(l2 == 0.0, 1.0, l2)          # avoid 0/0 at the DC mode
    d1 = (lx**2 - ly**2) / l2_safe
    d2 = 2.0 * lx * ly / l2_safe
    d1[0, 0] = 0.0
    d2[0, 0] = 0.0
    keep = _nyquist_keep(shape)
    d1 = d1 * keep
    d2 = d2 * keep
    d1.setflags(write=False)
    d2.setflags(write=False)
    return d1, d2


def forward_model(kappa_patch: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Convert a 2-D convergence map kappa -> (gamma1, gamma2) via FFTs (flat-sky Kaiser-Squires).

    Works on any 2-D shape. NaN/inf are replaced by 0. Returns float64 arrays with zero mean.
    """
    kappa = np.nan_to_num(np.asarray(kappa_patch, dtype=np.float64), nan=0.0, posinf=0.0, neginf=0.0)
    if kappa.ndim != 2:
        raise ValueError(f"kappa must be 2-D, got shape {kappa.shape}")
    d1, d2 = _ks_kernels(kappa.shape)
    k_ft = sfft.rfft2(kappa, workers=-1)
    gamma1 = sfft.irfft2(d1 * k_ft, s=kappa.shape, workers=-1)
    gamma2 = sfft.irfft2(d2 * k_ft, s=kappa.shape, workers=-1)
    return gamma1, gamma2


def kaiser_squires_inverse(gamma1: np.ndarray, gamma2: np.ndarray) -> np.ndarray:
    """Inverse of forward_model (up to the unrecoverable mean): kappa_ft = d1*g1_ft + d2*g2_ft."""
    d1, d2 = _ks_kernels(gamma1.shape)
    k_ft = d1 * sfft.rfft2(gamma1, workers=-1) + d2 * sfft.rfft2(gamma2, workers=-1)
    return sfft.irfft2(k_ft, s=gamma1.shape, workers=-1)

### Unit tests of the physics (run these once; they should all pass)

In [ ]:
# Test map: first real file if available, otherwise a synthetic smoothed Gaussian random field
if FITS_FILES:
    test_kappa = load_kappa(FITS_FILES[0]).astype(np.float64)
    print(f"Testing on {os.path.basename(FITS_FILES[0])}  shape={test_kappa.shape}")
else:
    from scipy.ndimage import gaussian_filter
    test_kappa = gaussian_filter(np.random.default_rng(0).normal(size=(512, 512)), 4) * 0.05
    print("No FITS files found -> testing on a synthetic Gaussian random field")

g1, g2 = forward_model(test_kappa)
sig = test_kappa.std()
# The forward model removes the DC mode (mean) and the Nyquist row/column; project the input the same way
keep = _nyquist_keep(test_kappa.shape).copy()
keep[0, 0] = 0.0
k0 = sfft.irfft2(sfft.rfft2(test_kappa) * keep, s=test_kappa.shape)
lost = 1 - np.sum(k0**2) / np.sum((test_kappa - test_kappa.mean()) ** 2)
print(f"       power in dropped Nyquist modes: {lost:.2e} of the total (variance)")

# 1. DC mode handled: no NaN, zero-mean shear
assert np.isfinite(g1).all() and np.isfinite(g2).all()
assert abs(g1.mean()) < 1e-12 and abs(g2.mean()) < 1e-12
print(f"[PASS] 1. DC mode OK: finite output, <g1>={g1.mean():.1e}, <g2>={g2.mean():.1e}")

# 2. Round trip kappa -> gamma -> kappa recovers the map up to its mean (and the dropped Nyquist modes)
err = np.max(np.abs(kaiser_squires_inverse(g1, g2) - k0))
assert err < 1e-9 * max(1.0, sig), err
print(f"[PASS] 2. Round trip: max|kappa_rec - kappa_projected| = {err:.2e}")

# 3. Power conservation: |gamma|^2 = |kappa|^2 mode by mode  =>  Parseval holds exactly
p_k, p_g = np.sum(k0**2), np.sum(g1**2 + g2**2)
assert abs(p_g / p_k - 1) < 1e-9, p_g / p_k
print(f"[PASS] 3. Power conservation: sum(g1^2+g2^2)/sum(kappa^2) = {p_g / p_k:.12f}")

# 4. (information) Edge effect of transforming a non-periodic patch vs. cutting from the full map
if min(test_kappa.shape) >= PATCH_SIZE and test_kappa.shape[0] > PATCH_SIZE:
    sl = center_slices(test_kappa.shape, PATCH_SIZE)
    g_cut = np.stack(forward_model(test_kappa))[(slice(None),) + sl]
    g_patch = np.stack(forward_model(test_kappa[sl]))
    rms = lambda a: float(np.sqrt(np.mean(a**2)))
    q = PATCH_SIZE // 4
    inner = (slice(None), slice(q, -q), slice(q, -q))
    print(f"[INFO] 4. Patch-wise vs full-map shear, relative RMS difference: "
          f"whole patch = {rms(g_patch - g_cut) / rms(g_cut):.1%},  central half = "
          f"{rms((g_patch - g_cut)[inner]) / rms(g_cut[inner]):.1%}")
    print("       -> Full-map shear avoids applying FFT periodicity to individual crops.")

## 3. Shape Noise & Full-Map Mask

**Shape noise:** independent Gaussian noise is added to each full-map shear component with standard deviation `NOISE_LEVEL` per pixel.

**Mask:** one full-width horizontal chip gap (height sampled in fixed pixels and clipped to no more than half the requested masked area), followed by circular star holes with radii sampled uniformly in fixed pixels until the requested masked fraction is reached. Pixel sizes do not scale with map dimensions. Returned mask is float32 (`1` valid, `0` missing); masked shear pixels are exactly zero.

In [ ]:
def add_shape_noise(gamma: np.ndarray, noise_level: float, rng: np.random.Generator | None = None) -> np.ndarray:
    """Add independent Gaussian shape noise to each shear component and pixel."""
    if noise_level < 0:
        raise ValueError("noise_level must be >= 0")
    rng = rng if rng is not None else np.random.default_rng()
    gamma = np.asarray(gamma, dtype=np.float64)
    return gamma + rng.normal(0.0, noise_level, size=gamma.shape)


def generate_mask(
    shape: tuple[int, int],
    mask_fraction: float,
    rng: np.random.Generator,
    star_radius_px: tuple[int, int],
    gap_height_px: tuple[int, int],
    max_circles: int = 5000,
) -> np.ndarray:
    """Build a full-map binary mask using a chip gap and circular star holes.

    The gap is capped at half the requested masked area. Each circle contributes
    only pixels that were previously valid, so overlap is not double-counted.
    Feature dimensions are fixed in pixels, independent of the map dimensions.
    """
    h, w = shape
    if h <= 0 or w <= 0:
        raise ValueError("mask shape dimensions must be positive")
    if not 0 <= mask_fraction < 1:
        raise ValueError("mask_fraction must be in [0, 1)")
    if len(star_radius_px) != 2 or star_radius_px[0] < 1 or star_radius_px[0] > star_radius_px[1]:
        raise ValueError("star_radius_px must be [lo, hi] with 1 <= lo <= hi")
    if len(gap_height_px) != 2 or gap_height_px[0] < 1 or gap_height_px[0] > gap_height_px[1]:
        raise ValueError("gap_height_px must be [lo, hi] with 1 <= lo <= hi")
    if max_circles <= 0:
        raise ValueError("max_circles must be positive")

    mask = np.ones(shape, dtype=np.float32)
    if mask_fraction == 0:
        return mask

    target_pixels = mask_fraction * h * w
    gap_cap = min(h, int(target_pixels * 0.5 // w))
    if gap_cap > 0:
        sampled_gap = int(rng.integers(gap_height_px[0], gap_height_px[1] + 1))
        gap_h = min(sampled_gap, gap_cap)
        y0 = int(rng.integers(0, h - gap_h + 1))
        mask[y0:y0 + gap_h, :] = 0.0

    n_masked = int(np.count_nonzero(mask == 0))
    n_circles = 0
    while n_masked < target_pixels and n_circles < max_circles:
        radius = float(rng.uniform(star_radius_px[0], np.nextafter(star_radius_px[1], np.inf)))
        cy, cx = float(rng.uniform(0, h)), float(rng.uniform(0, w))
        ys = slice(max(int(cy - radius), 0), min(int(cy + radius) + 2, h))
        xs = slice(max(int(cx - radius), 0), min(int(cx + radius) + 2, w))
        yy, xx = np.ogrid[ys, xs]
        inside = (yy - cy) ** 2 + (xx - cx) ** 2 <= radius**2
        region = mask[ys, xs]
        newly_masked = int(np.count_nonzero(region[inside]))
        n_masked += newly_masked
        region[inside] = 0.0
        n_circles += 1

    if n_masked < target_pixels:
        warnings.warn(
            f"Mask reached {n_masked / (h * w):.3%}, below requested {mask_fraction:.3%} "
            f"after {max_circles} circles.", RuntimeWarning,
        )
    return mask

In [ ]:
# Statistical mask test on the full 512x512 map (200 independent masks per level).
mask_test_rng = np.random.default_rng(SEED)
for fraction in (0.05, 0.15, 0.30):
    achieved = np.array([
        1 - generate_mask((512, 512), fraction, mask_test_rng,
                          STAR_RADIUS_PX, GAP_HEIGHT_PX).mean()
        for _ in range(200)
    ])
    assert abs(achieved.mean() - fraction) < 0.01, (fraction, achieved.mean())
    print(f"[PASS] mask {fraction:.0%}: mean={achieved.mean():.4f}, "
          f"range=[{achieved.min():.4f}, {achieved.max():.4f}]")

## 4. Full-Map Observation & Visual QA

For each source map, the order is fixed: full-map κ → full-map Kaiser–Squires shear → deterministic per-map Gaussian noise → deterministic full-map mask. The same saved `(x, y)` is later viewed as a full map, four fixed tiles, or random crops; crop selection never regenerates noise or masks.

In [ ]:
def compute_gamma_full(kappa_full: np.ndarray) -> np.ndarray:
    """Compute both Kaiser–Squires shear components on the complete map."""
    return np.stack(forward_model(kappa_full))


def make_observation(
    kappa_full: np.ndarray,
    noise_level: float,
    mask_fraction: float,
    rng: np.random.Generator,
    star_radius_px: tuple[int, int],
    gap_height_px: tuple[int, int],
    zero_mean: bool = True,
) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Create one full-map noisy/masked input, κ target, and clean shear pair.

    Returns x float32 (3,H,W), target float32 (H,W), and clean shear float64 (2,H,W).
    The optional target mean subtraction is performed over the full map before
    any data-loader crops are created.
    """
    kappa = np.asarray(kappa_full, dtype=np.float32)
    if kappa.ndim != 2:
        raise ValueError(f"kappa_full must be 2-D, got {kappa.shape}")
    gamma_clean = compute_gamma_full(kappa)
    gamma_noisy = add_shape_noise(gamma_clean, noise_level, rng)
    mask = generate_mask(kappa.shape, mask_fraction, rng, star_radius_px, gap_height_px)
    gamma_observed = gamma_noisy * mask[None, :, :]
    x = np.concatenate((gamma_observed, mask[None, :, :]), axis=0).astype(np.float32)
    target = kappa.copy()
    if zero_mean:
        target -= target.mean(dtype=np.float64).astype(np.float32)
    return np.ascontiguousarray(x), np.ascontiguousarray(target, dtype=np.float32), gamma_clean

In [ ]:
# Visual QA: random readable full maps, with the 256-pixel crop grid on the mask.
if not FITS_FILES:
    raise FileNotFoundError(f"No FITS files found under {INPUT_DIR}")

qa_rng = np.random.default_rng(SEED)
qa_order = qa_rng.permutation(len(FITS_FILES))
qa_rows = []
for index in qa_order:
    if len(qa_rows) >= N_QA_FILES:
        break
    path = FITS_FILES[int(index)]
    try:
        kappa = load_kappa(path)
        if kappa.shape[0] % PATCH_SIZE or kappa.shape[1] % PATCH_SIZE:
            raise ValueError(f"map shape {kappa.shape} is not divisible by PATCH_SIZE={PATCH_SIZE}")
        rng = np.random.default_rng([SEED, stable_seed_key(path)])
        x, target, gamma_clean = make_observation(
            kappa, NOISE_LEVEL, MASK_FRACTION, rng, STAR_RADIUS_PX, GAP_HEIGHT_PX, TARGET_ZERO_MEAN
        )
        qa_rows.append((path, x, target, gamma_clean))
    except Exception as exc:
        warnings.warn(f"QA skipped {os.path.basename(path)}: {type(exc).__name__}: {exc}")

if not qa_rows:
    raise RuntimeError("No readable map was available for visual QA.")

fig, axes = plt.subplots(len(qa_rows), 4, figsize=(18, 4.5 * len(qa_rows)), squeeze=False)
for row_axes, (path, x, target, gamma_clean) in zip(axes, qa_rows):
    valid = x[2] > 0
    if not valid.any():
        raise ValueError(f"Generated an all-masked map for {path}")
    shear_values = x[:2, valid]
    shear_limit = float(np.percentile(np.abs(shear_values), 99)) or 1.0
    meta = parse_metadata(path)
    masked = 1.0 - float(x[2].mean())
    title_meta = f"z={meta['redshift']:.2f}, masked={masked:.1%}"

    for ax, channel, label in zip(row_axes[:2], (0, 1), ("Noisy masked γ₁", "Noisy masked γ₂")):
        image = ax.imshow(x[channel], cmap="RdBu_r", origin="lower", vmin=-shear_limit, vmax=shear_limit)
        ax.set_title(f"{label}\n{title_meta}", fontsize=9)
        fig.colorbar(image, ax=ax, fraction=0.046)

    row_axes[2].imshow(x[2], cmap="gray", origin="lower", vmin=0, vmax=1)
    for edge in range(PATCH_SIZE, x.shape[1], PATCH_SIZE):
        row_axes[2].axvline(edge - 0.5, color="cyan", linestyle="--", linewidth=0.7)
    for edge in range(PATCH_SIZE, x.shape[2], PATCH_SIZE):
        row_axes[2].axhline(edge - 0.5, color="cyan", linestyle="--", linewidth=0.7)
    row_axes[2].set_title(f"Mask + tile grid\n{title_meta}", fontsize=9)

    target_lo, target_hi = np.percentile(target, [1, 99])
    image = row_axes[3].imshow(target, cmap="viridis", origin="lower", vmin=target_lo, vmax=target_hi)
    row_axes[3].set_title(f"Clean target κ\n{title_meta}", fontsize=9)
    fig.colorbar(image, ax=row_axes[3], fraction=0.046)
    for ax in row_axes:
        ax.set_xticks([])
        ax.set_yticks([])

fig.suptitle(f"Full-map QA · noise σ={NOISE_LEVEL}, target mask={MASK_FRACTION:.0%}", fontsize=13)
fig.tight_layout(rect=(0, 0, 1, 0.98))
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
qa_path = OUTPUT_DIR / QA_FIGURE
fig.savefig(qa_path, dpi=130, bbox_inches="tight")
print(f"Saved full-map QA figure: {qa_path.resolve()}")
plt.show()

**How to read the figure**
- The two shear panels share one symmetric colour scale; masking has zeroed invalid pixels.
- Cyan dashed lines on the mask mark the fixed tile boundaries. A horizontal gap will intersect only the tiles it crosses.
- Noise and masks are generated once per map and are reused by every loader mode.

## 5. Batch Processing & Full-Map Saving

Each readable FITS map produces one input/target pair stored as `.npy` members in `processed_maps.tar`. `manifest.csv` has one row per source map—not per crop—so future train/validation/test splits can keep all derived tiles together. The QA image, manifest, archive, and resolved config are grouped in `OUTPUT_DIR` (default: `generated_data/sample_1`). The original source TAR remains untouched.

In [ ]:
def _add_npy_member(archive: tarfile.TarFile, name: str, array: np.ndarray) -> None:
    """Add one reproducible NPY member to the output TAR archive."""
    buffer = io.BytesIO()
    np.save(buffer, array, allow_pickle=False)
    payload = buffer.getvalue()
    info = tarfile.TarInfo(name=name)
    info.size = len(payload)
    info.mtime = 0
    info.mode = 0o644
    info.uid = info.gid = 0
    info.uname = info.gname = ""
    archive.addfile(info, io.BytesIO(payload))


def _load_npy_member(archive: tarfile.TarFile, name: str) -> np.ndarray:
    """Load one NPY member without extracting the TAR archive."""
    member = archive.extractfile(name)
    if member is None:
        raise FileNotFoundError(f"Array member {name!r} is missing from {archive.name}")
    with member:
        return np.load(io.BytesIO(member.read()), allow_pickle=False)


def process_maps(
    file_paths: list[str],
    output_dir: Path,
    noise_level: float,
    mask_fraction: float,
    seed: int,
    patch_size: int,
    star_radius_px: tuple[int, int],
    gap_height_px: tuple[int, int],
    target_zero_mean: bool,
) -> tuple[list[dict], list[tuple[str, str]]]:
    """Generate deterministic full-map pairs and package them in processed_maps.tar."""
    output_dir.mkdir(parents=True, exist_ok=True)
    legacy = sorted(output_dir.glob("input_[0-9]*.npy")) + sorted(output_dir.glob("target_[0-9]*.npy"))
    if legacy:
        warnings.warn(
            f"{len(legacy)} legacy numbered patch file(s) exist in {output_dir}; they are not deleted or reused.",
            RuntimeWarning,
        )

    manifest_rows: list[dict] = []
    failed_files: list[tuple[str, str]] = []
    output_stems: dict[str, str] = {}
    archive_path = output_dir / "processed_maps.tar"
    iterator = tqdm(file_paths, desc="Processing maps", unit="map") if tqdm else file_paths

    with tarfile.open(archive_path, mode="w") as output_archive:
        for index, path in enumerate(iterator, start=1):
            source_name = os.path.basename(path)
            stem = Path(source_name).stem
            source_relative = source_name
            try:
                try:
                    source_relative = Path(path).resolve().relative_to(INPUT_DIR.resolve()).as_posix()
                except ValueError:
                    source_relative = Path(path).name
                if stem in output_stems:
                    raise ValueError(
                        f"Output stem collision: {source_relative!r} and {output_stems[stem]!r} both map to {stem!r}"
                    )
                kappa = load_kappa(path)
                height, width = kappa.shape
                if height % patch_size or width % patch_size:
                    raise ValueError(f"map shape {kappa.shape} is not divisible by PATCH_SIZE={patch_size}")

                file_seed_key = stable_seed_key(path)
                rng = np.random.default_rng([seed, file_seed_key])
                x, target, _ = make_observation(
                    kappa, noise_level, mask_fraction, rng,
                    star_radius_px, gap_height_px, target_zero_mean,
                )
                input_name = f"{stem}_input.npy"
                target_name = f"{stem}_target.npy"
                _add_npy_member(output_archive, input_name, x)
                _add_npy_member(output_archive, target_name, target)
                output_stems[stem] = source_relative
                meta = parse_metadata(path)
                mean_removed = float(np.mean(kappa, dtype=np.float64)) if target_zero_mean else 0.0
                manifest_rows.append({
                    "source_fits": source_relative,
                    "redshift": meta["redshift"],
                    "realization": meta["realization"],
                    "input_file": input_name,
                    "target_file": target_name,
                    "height": height,
                    "width": width,
                    "noise_level": noise_level,
                    "mask_fraction_target": mask_fraction,
                    "mask_fraction_achieved": float(1.0 - x[2].mean()),
                    "seed": seed,
                    "file_seed_key": file_seed_key,
                    "target_mean_removed": mean_removed,
                })
            except Exception as exc:
                message = f"{type(exc).__name__}: {exc}"
                warnings.warn(f"FAILED {source_relative}: {message}", RuntimeWarning)
                failed_files.append((source_relative, message))

            if tqdm is None and (index % 10 == 0 or index == len(file_paths)):
                print(f"[{index}/{len(file_paths)}] maps visited, {len(manifest_rows)} saved, {len(failed_files)} skipped")

    columns = [
        "source_fits", "redshift", "realization", "input_file", "target_file",
        "height", "width", "noise_level", "mask_fraction_target",
        "mask_fraction_achieved", "seed", "file_seed_key", "target_mean_removed",
    ]
    with (output_dir / "manifest.csv").open("w", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        writer.writerows(manifest_rows)

    config_to_save = {key: (str(value) if isinstance(value, Path) else value) for key, value in cfg_used.items()}
    config_to_save.update({
        "INPUT_DIR_RESOLVED": str(INPUT_DIR.resolve()),
        "OUTPUT_DIR_RESOLVED": str(output_dir.resolve()),
        "OUTPUT_ARCHIVE": archive_path.name,
        "MAPS_SUCCEEDED": len(manifest_rows),
        "MAPS_FAILED": len(failed_files),
    })
    with (output_dir / "config_used.yaml").open("w", encoding="utf-8") as stream:
        yaml.safe_dump(config_to_save, stream, sort_keys=False)

    print(f"\nDone: {len(manifest_rows)}/{len(file_paths)} maps saved to {archive_path.resolve()}")
    print(f"Manifest: {output_dir / 'manifest.csv'}")
    return manifest_rows, failed_files


manifest, failed = process_maps(
    FITS_FILES, OUTPUT_DIR, NOISE_LEVEL, MASK_FRACTION, SEED,
    PATCH_SIZE, STAR_RADIUS_PX, GAP_HEIGHT_PX, TARGET_ZERO_MEAN,
)
if failed:
    print("Skipped maps:")
    for filename, reason in failed:
        print(f"  {filename}: {reason}")

### Verify every saved full-map pair from the generated TAR archive

In [ ]:
manifest_path = OUTPUT_DIR / "manifest.csv"
archive_path = OUTPUT_DIR / "processed_maps.tar"
with manifest_path.open(newline="", encoding="utf-8") as stream:
    saved_rows = list(csv.DictReader(stream))

bad_pairs = []
mask_fractions = []
with tarfile.open(archive_path, mode="r:") as saved_archive:
    for row in saved_rows:
        x = _load_npy_member(saved_archive, row["input_file"])
        y = _load_npy_member(saved_archive, row["target_file"])
        h, w = int(row["height"]), int(row["width"])
        valid = (
            x.shape == (3, h, w) and x.dtype == np.float32
            and y.shape == (h, w) and y.dtype == np.float32
            and np.isfinite(x).all() and np.isfinite(y).all()
            and set(np.unique(x[2])).issubset({0.0, 1.0})
            and np.all(x[:2, x[2] == 0] == 0.0)
        )
        if not valid:
            bad_pairs.append(row["source_fits"])
        mask_fractions.append(float(1.0 - x[2].mean()))

if bad_pairs:
    raise AssertionError(f"Saved pair contract violations: {bad_pairs}")
print(f"Verified {len(saved_rows)} full-map pair(s) inside {archive_path.name}: shapes, dtypes, finite values, binary masks, masked shear zeros PASS")
if mask_fractions:
    print(f"Mask fraction: mean={np.mean(mask_fractions):.4f}, min={np.min(mask_fractions):.4f}, max={np.max(mask_fractions):.4f}")
redshift_counts = {}
for row in saved_rows:
    key = row["redshift"]
    redshift_counts[key] = redshift_counts.get(key, 0) + 1
print("Maps per redshift:", redshift_counts)
print(f"Generated archive size: {archive_path.stat().st_size / (1024**2):.2f} MiB")
print(f"Manifest rows: {len(saved_rows)}; failed source maps: {len(failed)}")

## 6. Dataset Loader: Full Maps, Fixed Tiles, and Random Crops

`MapCropDataset` reads one-per-map arrays directly from `processed_maps.tar` and exposes three views over identical generated noise/masks: `full`, row-major non-overlapping `tiles`, and deterministic `random` crops. Random crop offsets change after `set_epoch(epoch)`. Use `augment=True` **only for training data**; validation/test datasets should use `augment=False`. An optional split CSV filters by `source_fits` and errors if any requested ID is absent. With `preload=True`, arrays are loaded once into RAM; with `preload=False`, the needed pair is read from the archive on demand. When PyTorch is unavailable, items contain equivalent float32 NumPy arrays.

In [ ]:
try:
    import torch
    from torch.utils.data import Dataset as _TorchDataset
except ImportError:  # NumPy-only preprocessing remains usable without PyTorch.
    torch = None
    _TorchDataset = object


def _numpy_array(value) -> np.ndarray:
    """Convert a NumPy array or CPU/GPU torch tensor to a NumPy view for stitching/tests."""
    if torch is not None and isinstance(value, torch.Tensor):
        return value.detach().cpu().numpy()
    return np.asarray(value)


def apply_dihedral_primitive(x: np.ndarray, y: np.ndarray, operation: str) -> tuple[np.ndarray, np.ndarray]:
    """Apply one spatial symmetry with the correct spin-2 shear component signs."""
    x_out = np.asarray(x).copy()
    y_out = np.asarray(y).copy()
    if operation == "flip_x":
        x_out = np.flip(x_out, axis=-1).copy()
        y_out = np.flip(y_out, axis=-1).copy()
        x_out[1] *= -1
    elif operation == "flip_y":
        x_out = np.flip(x_out, axis=-2).copy()
        y_out = np.flip(y_out, axis=-2).copy()
        x_out[1] *= -1
    elif operation == "transpose":
        x_out = np.swapaxes(x_out, -2, -1).copy()
        y_out = np.swapaxes(y_out, -2, -1).copy()
        x_out[0] *= -1
    else:
        raise ValueError(f"Unknown dihedral operation: {operation!r}")
    return x_out, y_out


def stitch_tiles(tiles, positions: list[tuple[int, int]], full_shape: tuple[int, ...]):
    """Reassemble non-overlapping tile arrays into a full spatial array.

    `tiles` is (N,H,W) or (N,C,H,W); `full_shape` is (H,W) or (C,H,W).
    Every output pixel must be covered exactly once.
    """
    tile_array = _numpy_array(tiles)
    if tile_array.ndim not in (3, 4) or len(positions) != len(tile_array):
        raise ValueError("tiles must have shape (N,H,W) or (N,C,H,W), with one position per tile")
    full_shape = tuple(int(v) for v in full_shape)
    expected_ndim = tile_array.ndim - 1
    if len(full_shape) != expected_ndim:
        raise ValueError(f"full_shape must have {expected_ndim} dimensions for these tiles")
    spatial_shape = full_shape[-2:]
    out = np.zeros(full_shape, dtype=tile_array.dtype)
    covered = np.zeros(spatial_shape, dtype=bool)
    for tile, (y0, x0) in zip(tile_array, positions):
        height, width = tile.shape[-2:]
        if y0 < 0 or x0 < 0 or y0 + height > spatial_shape[0] or x0 + width > spatial_shape[1]:
            raise ValueError(f"tile at {(y0, x0)} lies outside full shape {spatial_shape}")
        region = (slice(y0, y0 + height), slice(x0, x0 + width))
        if covered[region].any():
            raise ValueError(f"overlapping tile at {(y0, x0)}")
        out[(...,) + region] = tile
        covered[region] = True
    if not covered.all():
        raise ValueError(f"tiles leave {int((~covered).sum())} full-map pixels uncovered")
    return out


class MapCropDataset(_TorchDataset):
    """Full-map FITS-derived arrays exposed as full maps, fixed tiles, or epoch crops."""

    def __init__(
        self,
        data_dir,
        manifest_csv,
        split_csv=None,
        mode="tiles",
        patch_size=256,
        crops_per_map=4,
        augment=False,
        crop_zero_mean=False,
        seed=0,
        preload=True,
    ):
        if mode not in {"full", "tiles", "random"}:
            raise ValueError("mode must be 'full', 'tiles', or 'random'")
        if patch_size <= 0 or crops_per_map <= 0:
            raise ValueError("patch_size and crops_per_map must be positive")
        self.data_dir = Path(data_dir)
        self.manifest_csv = Path(manifest_csv)
        self.mode = mode
        self.patch_size = int(patch_size)
        self.crops_per_map = int(crops_per_map)
        self.augment = bool(augment)
        self.crop_zero_mean = bool(crop_zero_mean)
        self.seed = int(seed)
        self.epoch = 0
        with self.manifest_csv.open(newline="", encoding="utf-8") as stream:
            rows = list(csv.DictReader(stream))
        if not rows:
            raise ValueError(f"Manifest has no map rows: {self.manifest_csv}")

        if split_csv is not None:
            with Path(split_csv).open(newline="", encoding="utf-8") as stream:
                split_rows = list(csv.DictReader(stream))
            if not split_rows or "source_fits" not in split_rows[0]:
                raise ValueError("split_csv must contain a source_fits column")
            selected_ids = {row["source_fits"] for row in split_rows}
            manifest_ids = {row["source_fits"] for row in rows}
            missing = sorted(selected_ids - manifest_ids)
            if missing:
                raise ValueError(f"split_csv source_fits IDs missing from manifest: {missing}")
            rows = [row for row in rows if row["source_fits"] in selected_ids]
        if not rows:
            raise ValueError("No manifest maps remain after applying split_csv")

        self.rows = rows
        self.preload = bool(preload)
        self._arrays = {}
        if self.preload:
            for index, row in enumerate(self.rows):
                self._arrays[index] = self._load_pair(row)

        sample_x, sample_y = self._get_pair(0)
        self.map_shape = tuple(sample_y.shape)
        if len(self.map_shape) != 2 or self.map_shape[0] % self.patch_size or self.map_shape[1] % self.patch_size:
            raise ValueError(f"map shape {self.map_shape} must be divisible by patch_size={self.patch_size}")
        if sample_x.shape != (3, *self.map_shape) or sample_x.dtype != np.float32 or sample_y.dtype != np.float32:
            raise ValueError("saved arrays must be float32 with x=(3,H,W), y=(H,W)")
        self.tiles_y = self.map_shape[0] // self.patch_size
        self.tiles_x = self.map_shape[1] // self.patch_size

    def _load_pair(self, row):
        archive_path = self.data_dir / "processed_maps.tar"
        with tarfile.open(archive_path, mode="r:") as archive:
            x = _load_npy_member(archive, row["input_file"])
            y = _load_npy_member(archive, row["target_file"])
        return np.asarray(x, dtype=np.float32), np.asarray(y, dtype=np.float32)

    def _get_pair(self, map_index):
        if self.preload:
            return self._arrays[map_index]
        return self._load_pair(self.rows[map_index])

    def set_epoch(self, epoch):
        """Set random-crop and augmentation epoch for deterministic worker-safe streams."""
        self.epoch = int(epoch)

    def __len__(self):
        if self.mode == "full":
            return len(self.rows)
        if self.mode == "tiles":
            return len(self.rows) * self.tiles_y * self.tiles_x
        return len(self.rows) * self.crops_per_map

    def __getitem__(self, index):
        if index < 0 or index >= len(self):
            raise IndexError(index)
        if self.mode == "full":
            map_index, y0, x0 = index, 0, 0
        elif self.mode == "tiles":
            tiles_per_map = self.tiles_y * self.tiles_x
            map_index, tile_index = divmod(index, tiles_per_map)
            tile_y, tile_x = divmod(tile_index, self.tiles_x)
            y0, x0 = tile_y * self.patch_size, tile_x * self.patch_size
        else:
            map_index, _ = divmod(index, self.crops_per_map)
            rng = np.random.default_rng([self.seed, self.epoch, index])
            y0 = int(rng.integers(0, self.map_shape[0] - self.patch_size + 1))
            x0 = int(rng.integers(0, self.map_shape[1] - self.patch_size + 1))

        x_full, y_full = self._get_pair(map_index)
        if self.mode == "full":
            x = x_full.copy()
            y = y_full[None, ...].copy()
        else:
            y1, x1 = y0 + self.patch_size, x0 + self.patch_size
            x = x_full[:, y0:y1, x0:x1].copy()
            y = y_full[None, y0:y1, x0:x1].copy()
        if self.crop_zero_mean:
            y -= y.mean(dtype=np.float64).astype(np.float32)

        if self.augment:
            rng = np.random.default_rng([self.seed, self.epoch, index, 0xA09])
            for operation in ("flip_x", "flip_y", "transpose"):
                if rng.random() < 0.5:
                    x, y = apply_dihedral_primitive(x, y, operation)

        if torch is not None:
            x_item, y_item = torch.from_numpy(np.ascontiguousarray(x)), torch.from_numpy(np.ascontiguousarray(y))
        else:
            x_item, y_item = np.ascontiguousarray(x), np.ascontiguousarray(y)
        return {
            "x": x_item,
            "y": y_item,
            "source_fits": self.rows[map_index]["source_fits"],
            "redshift": float(self.rows[map_index]["redshift"]),
            "y0": y0,
            "x0": x0,
        }

### Self-checks

These checks run on temporary synthetic nested FITS maps, including one NaN-valued map and one corrupt FITS file. They verify the processed TAR member contract, reproducibility, loader modes/stitching, split validation, augmentation equivariance, corruption handling, and retained Kaiser–Squires physics invariants. No synthetic outputs are written to the real dataset directory.

In [ ]:
import csv as _csv
import hashlib as _hashlib
import tempfile as _tempfile
import warnings as _warnings


def _sha256(path):
    digest = _hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


def _synthetic_fits(path, seed, add_nan=False):
    path.parent.mkdir(parents=True, exist_ok=True)
    rng = np.random.default_rng(seed)
    from scipy.ndimage import gaussian_filter
    image = (gaussian_filter(rng.normal(size=(32, 32)), 2) * 0.03).astype(np.float32)
    if add_nan:
        image[3, 7] = np.nan
    fits.PrimaryHDU(image).writeto(path, overwrite=True)


with _tempfile.TemporaryDirectory(prefix="cosmicweb-pipeline-check-") as temp:
    temp_root = Path(temp)
    synthetic_input = temp_root / "input"
    paths = [
        synthetic_input / "nested_a" / "WLconv_z0.50_1001r.fits",
        synthetic_input / "nested_b" / "WLconv_z0.50_1002r.fits",
        synthetic_input / "nested_c" / "WLconv_z1.00_1003r.fits",
    ]
    for i, path in enumerate(paths):
        _synthetic_fits(path, i + 10, add_nan=(i == 1))
    corrupt_path = synthetic_input / "nested_bad" / "WLconv_z1.00_9999r.fits"
    corrupt_path.parent.mkdir(parents=True)
    corrupt_path.write_bytes(b"not a FITS file")
    synthetic_paths = [str(path) for path in paths] + [str(corrupt_path)]

    test_output = temp_root / "processed"
    with _warnings.catch_warnings(record=True) as caught:
        _warnings.simplefilter("always")
        check_manifest, check_failed = process_maps(
            synthetic_paths, test_output, 0.01, 0.15, 73, 16,
            (2, 4), (1, 2), True,
        )
    assert len(check_manifest) == 3 and len(check_failed) == 1
    assert any("non-finite pixels set to 0" in str(w.message) for w in caught)
    assert any("FAILED" in str(w.message) and "9999r" in str(w.message) for w in caught)
    print("[PASS] corrupt FITS skipped with warning; NaN FITS processed with warning")

    # Saved pair contract and full-map filenames.
    with tarfile.open(test_output / "processed_maps.tar", mode="r:") as check_archive:
        for row in check_manifest:
            x = _load_npy_member(check_archive, row["input_file"])
            y = _load_npy_member(check_archive, row["target_file"])
            assert x.shape == (3, 32, 32) and x.dtype == np.float32
            assert y.shape == (32, 32) and y.dtype == np.float32
            assert np.isfinite(x).all() and np.isfinite(y).all()
            assert set(np.unique(x[2])).issubset({0.0, 1.0})
            assert np.all(x[:2, x[2] == 0] == 0.0)
    assert check_manifest[0]["input_file"] == "WLconv_z0.50_1001r_input.npy"
    print("[PASS] generated TAR contains the expected full-map arrays and valid saved pair shapes/dtypes/masks")

    # A second batch run must recreate a byte-identical TAR archive.
    archive_path = test_output / "processed_maps.tar"
    hash_before = _sha256(archive_path)
    with _warnings.catch_warnings():
        _warnings.simplefilter("ignore")
        process_maps(synthetic_paths, test_output, 0.01, 0.15, 73, 16, (2, 4), (1, 2), True)
    hash_after = _sha256(archive_path)
    assert hash_before == hash_after
    print("[PASS] repeated batch run produces a byte-identical processed_maps.tar (SHA-256)")

    # Loader modes and exact tile stitching from the stored full map.
    manifest_csv = test_output / "manifest.csv"
    full_ds = MapCropDataset(test_output, manifest_csv, mode="full", patch_size=16, preload=True)
    tiles_ds = MapCropDataset(test_output, manifest_csv, mode="tiles", patch_size=16, preload=True)
    random_ds = MapCropDataset(test_output, manifest_csv, mode="random", patch_size=16,
                               crops_per_map=3, preload=True, seed=5)
    assert len(full_ds) == 3 and len(tiles_ds) == 12 and len(random_ds) == 9
    assert tuple(full_ds[0]["x"].shape) == (3, 32, 32)
    assert tuple(tiles_ds[0]["x"].shape) == (3, 16, 16)
    assert tuple(tiles_ds[0]["y"].shape) == (1, 16, 16)
    assert tuple(random_ds[0]["y"].shape) == (1, 16, 16)
    random_first = (random_ds[0]["y0"], random_ds[0]["x0"])
    random_ds.set_epoch(1)
    random_epoch1 = (random_ds[0]["y0"], random_ds[0]["x0"])
    random_ds.set_epoch(0)
    assert random_first == (random_ds[0]["y0"], random_ds[0]["x0"])
    # With the chosen seed, changing epoch should select a different crop.
    assert random_first != random_epoch1
    first_full = full_ds[0]
    tile_items = [tiles_ds[i] for i in range(4)]
    tile_positions = [(item["y0"], item["x0"]) for item in tile_items]
    stitched_x = stitch_tiles(np.stack([_numpy_array(item["x"]) for item in tile_items]), tile_positions, (3, 32, 32))
    stitched_y = stitch_tiles(np.stack([_numpy_array(item["y"])[0] for item in tile_items]), tile_positions, (32, 32))
    np.testing.assert_array_equal(stitched_x, _numpy_array(first_full["x"]))
    np.testing.assert_array_equal(stitched_y, _numpy_array(first_full["y"])[0])
    print("[PASS] loader lengths, shapes, epoch-random crops, and exact four-tile full-map stitching")

    # Unknown split identity must fail explicitly.
    bad_split = temp_root / "unknown_split.csv"
    with bad_split.open("w", newline="", encoding="utf-8") as stream:
        writer = _csv.DictWriter(stream, fieldnames=["source_fits"])
        writer.writeheader()
        writer.writerow({"source_fits": "missing/map.fits"})
    try:
        MapCropDataset(test_output, manifest_csv, split_csv=bad_split, mode="full", patch_size=16)
    except ValueError as exc:
        assert "missing from manifest" in str(exc)
    else:
        raise AssertionError("unknown split source_fits did not raise ValueError")
    print("[PASS] unknown split source_fits raises a clear ValueError")

    # Exact shear equivariance for each generator and random compositions.
    yy, xx = np.mgrid[:32, :32]
    kappa = (np.sin(xx / 4.3) + 0.7 * np.cos(yy / 5.1) + 0.2 * np.sin((xx + yy) / 3.7)).astype(np.float64)
    def check_equivariance(kappa_map, operations):
        shear = np.stack(forward_model(kappa_map))
        sample_x = np.concatenate((shear, np.ones((1, *kappa_map.shape))), axis=0)
        sample_y = kappa_map[None]
        expected_kappa = kappa_map.copy()
        transformed_x, transformed_y = sample_x, sample_y
        for operation in operations:
            transformed_kappa = (
                np.flip(expected_kappa, axis=-1).copy() if operation == "flip_x" else
                np.flip(expected_kappa, axis=-2).copy() if operation == "flip_y" else
                np.swapaxes(expected_kappa, -2, -1).copy()
            )
            transformed_x, transformed_y = apply_dihedral_primitive(transformed_x, transformed_y, operation)
            expected_kappa = transformed_kappa
        actual_shear = np.stack(forward_model(expected_kappa))
        assert np.max(np.abs(transformed_x[:2] - actual_shear)) < 1e-9
        np.testing.assert_array_equal(transformed_y[0], expected_kappa)

    for operation in ("flip_x", "flip_y", "transpose"):
        check_equivariance(kappa, [operation])
    composition_rng = np.random.default_rng(101)
    operations = ("flip_x", "flip_y", "transpose")
    for _ in range(8):
        composition = [operation for operation in operations if composition_rng.random() < 0.5]
        check_equivariance(kappa, composition)
    print("[PASS] shear spin-2 equivariance for flip_x, flip_y, transpose, and random compositions (<1e-9)")

    # Retain original DC, inverse round-trip, and power conservation checks.
    kappa_test = np.random.default_rng(102).normal(size=(32, 32))
    g1_test, g2_test = forward_model(kappa_test)
    keep = _nyquist_keep(kappa_test.shape).copy()
    keep[0, 0] = 0.0
    kappa_projected = sfft.irfft2(sfft.rfft2(kappa_test) * keep, s=kappa_test.shape)
    assert np.isfinite(g1_test).all() and np.isfinite(g2_test).all()
    assert abs(g1_test.mean()) < 1e-12 and abs(g2_test.mean()) < 1e-12
    assert np.max(np.abs(kaiser_squires_inverse(g1_test, g2_test) - kappa_projected)) < 1e-9
    power_ratio = np.sum(g1_test**2 + g2_test**2) / np.sum(kappa_projected**2)
    assert abs(power_ratio - 1.0) < 1e-9
    print("[PASS] original physics tests: DC handling, inverse round-trip, and power conservation")

print("\\nALL SELF-CHECKS PASSED")

## Notes & Next Steps

1. **Output layout:** `OUTPUT_DIR` contains `processed_maps.tar` (all generated per-map NPY arrays), `manifest.csv`, `qa_forward_model.png`, and `config_used.yaml`. The original input TAR is never modified.
2. **Leakage-safe splits:** create train/validation/test manifests grouped by `source_fits` before generating any derived tile records. All views from a source map must stay in one split.
3. **Identical experiment inputs:** `full`, `tiles`, and `random` loader modes read the same persisted full-map input from the archive; they do not regenerate noise or masks. Random positions vary by epoch through `set_epoch`.
4. **Augmentation:** instantiate training datasets with `augment=True`; instantiate validation and test datasets with `augment=False`. Spin-2 sign changes are handled during flips/transposes.
5. **Training targets:** loader items return `y` with shape `(1,H,W)` or `(1,P,P)`. `crop_zero_mean=True` optionally removes the selected crop mean; default is false.
6. **Memory:** `preload=True` uses about 4 MiB per 512×512 map pair (3 input channels plus one target channel); use `preload=False` for larger data or limited RAM.
7. **Noise experiments:** change `NOISE_LEVEL` and `OUTPUT_DIR` in YAML for separate low/mid/high runs; preserve the config and manifest with each result.